# 05 — Robustesse : le système sait-il quand l'image est dégradée ?

**Rôle** : dégrader progressivement les images (compression JPEG, bruit, flou, recadrage, luminosité) et observer trois choses :
1. l'indice de fiabilité **baisse-t-il** quand l'image se dégrade ?
2. la part de **refus** augmente-t-elle ?
3. l'AUROC de cas **conservés** reste-t-elle correcte ?

Un système robuste n'est pas celui qui ne se trompe jamais sur des images dégradées, c'est celui qui **le remarque**.

In [ ]:
# --- Initialisation (identique dans tous les notebooks) ---
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
IN_KAGGLE = Path("/kaggle").exists()
if IN_KAGGLE:
    # Internet doit être ACTIVÉ dans les réglages du notebook Kaggle.
    # Remplacez VOTRE_COMPTE par votre identifiant GitHub (dépôt medicure-ai).
    %pip install -q git+https://github.com/VOTRE_COMPTE/medicure-ai.git torchxrayvision
else:
    sys.path.insert(0, str(Path.cwd().parent))  # notebook lancé depuis le dossier notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from medguard.config import TARGETS, MODEL_WEIGHTS, Settings, load_settings, save_settings
from medguard.paths import find_file

# --- Configuration (à adapter) ---
DATASET = os.environ.get("MEDGUARD_DATASET", "nih")            # "nih" ou "chexpert"
DATA_ROOT = Path(os.environ.get("MEDGUARD_DATA", "/kaggle/input/data"))  # À ADAPTER au dataset attaché
WEIGHTS = os.environ.get("MEDGUARD_WEIGHTS", MODEL_WEIGHTS)
N_MAX = int(os.environ.get("MEDGUARD_NMAX", 6000))               # nb max d'images chargées
SEED = 42

WORK = Path("/kaggle/working") if IN_KAGGLE else Path(os.environ.get("MEDGUARD_WORK", "../outputs")).resolve()
ARTIFACTS = WORK / "artifacts"
FIGURES = WORK / "figures"
for d in (ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
print("Kaggle :", IN_KAGGLE, "| dataset :", DATASET, "| poids :", WEIGHTS, "| sortie :", WORK)

In [ ]:
import json
from medguard.calibration import Calibrator
from medguard.data import load_dataset
from medguard.evaluation import auroc
from medguard.imaging import load_image, to_gray
from medguard.model import get_predictor
from medguard.pipeline import MedGuardPipeline
from medguard.preprocess import apply_perturbation
from medguard.splits import load_splits
from PIL import Image

N_ROBUST = int(os.environ.get("MEDGUARD_NROBUST", 150))

df, EVAL_TARGETS = load_dataset(DATASET, DATA_ROOT, n_max=N_MAX, seed=SEED)
splits = load_splits(find_file("splits.json", [WORK, ARTIFACTS]))
df = df[df.image_id.astype(str).isin(set(splits["test"]))].reset_index(drop=True)
df = df.sample(min(len(df), N_ROBUST), random_state=SEED).reset_index(drop=True)

try:
    settings = load_settings(find_file("thresholds.json", [ARTIFACTS, WORK]))
except FileNotFoundError:
    settings = load_settings()
calibrator = Calibrator.load(find_file("calibration.json", [ARTIFACTS, WORK]))
pipe = MedGuardPipeline(get_predictor(WEIGHTS), calibrator=calibrator, settings=settings)
grays = [to_gray(load_image(p)) for p in df.path]
print(len(grays), "images de test")

In [ ]:
from tqdm.auto import tqdm

GRID = {
    "jpeg":       [95, 70, 40, 15],          # qualité JPEG (plus bas = pire)
    "noise":      [2, 6, 12, 24],            # écart-type du bruit
    "blur":       [0.5, 1.5, 3.0, 6.0],      # sigma du flou gaussien
    "crop":       [0.02, 0.06, 0.12, 0.2],   # marge rognée de chaque côté
    "brightness": [1.15, 1.4, 0.6, 0.35],    # facteur de luminosité
}

def run(kind, strength):
    rel, status, probs = [], [], {t: [] for t in EVAL_TARGETS}
    for g in grays:
        arr = g if kind is None else apply_perturbation(g, kind, strength)
        res = pipe.analyze(Image.fromarray(arr), with_explanations=False)
        rel.append(res.reliability.index if res.reliability else np.nan)
        status.append(res.status)
        got = {t.name: t.calibrated for t in res.targets} if res.targets else {}
        for t in EVAL_TARGETS:
            probs[t].append(got.get(t, np.nan))
    rel, status = np.array(rel), np.array(status)
    row = {"perturbation": kind or "aucune", "force": strength, "fiabilité moyenne": np.nanmean(rel),
           "% refus ou hors périmètre": float((status != "ok").mean() * 100)}
    for t in EVAL_TARGETS:
        y = df[t].values.astype(float); p = np.array(probs[t]); ok = ~np.isnan(y) & ~np.isnan(p)
        row[f"AUROC {t} (cas conservés)"] = auroc(y[ok], p[ok]) if ok.sum() > 10 else np.nan
    return row

rows = [run(None, 0)]
for kind, strengths in GRID.items():
    for s in tqdm(strengths, desc=kind):
        rows.append(run(kind, s))
robust = pd.DataFrame(rows)
robust.to_csv(ARTIFACTS / "robustness.csv", index=False)
robust.round(3)

In [ ]:
kinds = list(GRID)
fig, axes = plt.subplots(2, len(kinds), figsize=(3.6 * len(kinds), 6.2), sharey="row")
base = robust[robust.perturbation == "aucune"].iloc[0]
for c, kind in enumerate(kinds):
    sub = robust[robust.perturbation == kind]
    x = range(len(sub)); labels = [str(v) for v in sub["force"]]
    axes[0, c].plot(x, sub["fiabilité moyenne"], "o-", color="#4b7bec"); axes[0, c].axhline(base["fiabilité moyenne"], ls="--", color="grey")
    axes[0, c].set_title(kind); axes[0, c].set_xticks(list(x)); axes[0, c].set_xticklabels(labels)
    axes[1, c].bar(x, sub["% refus ou hors périmètre"], color="#eb3b5a"); axes[1, c].set_xticks(list(x)); axes[1, c].set_xticklabels(labels)
    axes[1, c].set_xlabel("force de la dégradation")
axes[0, 0].set_ylabel("fiabilité moyenne (0-100)"); axes[1, 0].set_ylabel("% refus / hors périmètre")
plt.tight_layout(); plt.savefig(FIGURES / "05_robustness.png", dpi=160); plt.show()

### Lecture
- **Ce qu'on veut voir** : la fiabilité moyenne **descend** et le taux de refus **monte** quand la dégradation augmente. C'est la preuve que le système remarque quand il est aveugle.
- **Ce qui serait un problème** : une dégradation forte avec une fiabilité inchangée (le système est confiant à tort) : à signaler comme limite, et à corriger (ajouter une métrique de qualité, resserrer les seuils).
- Le recadrage et la luminosité sont des dégradations **discutables** : une radio un peu plus claire n'est pas forcément « mauvaise ». Interprétez avec l'œil d'un clinicien.